# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [49]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [50]:
# Initialization

load_dotenv(override=True)

gemini_api_key = os.getenv('GEMINI_API_KEY')
openrouter_paid_api_key = os.getenv('OPENROUTER_PAID_API_KEY')
if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:4]}")
else:
    print("Gemini API Key not set")
if openrouter_paid_api_key:
    print(f"paid  API Key exists and begins {openrouter_paid_api_key[:4]}")
else:
    print("openrouter paid API Key not set")

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
op_url = "https://openrouter.ai/api/v1"
    
MODEL = "gemini-2.5-flash"
gemini = OpenAI(base_url=gemini_url ,api_key=gemini_api_key)
openrouter = OpenAI(base_url=op_url,api_key=openrouter_paid_api_key)

DB = "prices.db"

Gemini API Key exists and begins AIza
paid  API Key exists and begins sk-o


In [51]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [52]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("Paris")

In [53]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [ ]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [54]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat).launch()

## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this React app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use DALL-E-3, the image generation model behind GPT-4o, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 4 cents - don't go crazy with images!

In [47]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [48]:
import base64
import requests
from io import BytesIO
from PIL import Image


def artist(city):
    prompt_text = (
        f"An image representing a vacation in {city}, showing tourist spots "
        f"and everything unique about {city}, in a vibrant pop-art style"
    )

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {openrouter_paid_api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": "google/gemini-2.5-flash-image",
            "messages": [
                {
                    "role": "user",
                    "content": [{"type": "text", "text": prompt_text}],
                }
            ],
            "modalities": ["image", "text"],
        },
    )

    response.raise_for_status()
    data = response.json()

    # Image lives in message["images"], not message["content"]
    message = data["choices"][0]["message"]
    images = message.get("images", [])

    if not images:
        raise ValueError(f"No images returned. Message content: {message.get('content')}")

    # Each entry: {"type": "image_url", "image_url": {"url": "data:image/png;base64,..."}}
    data_url = images[0]["image_url"]["url"]
    _, b64 = data_url.split(",", 1)
    return Image.open(BytesIO(base64.b64decode(b64)))



In [ ]:
image = artist("New York City")
display(image)

In [55]:
import requests
from IPython.display import Audio, display


def tts(text, voice="af_heart", output_file="speech.mp3"):
    """
    Convert text to speech using Kokoro 82M via OpenRouter.
    
    Cheapest TTS on OpenRouter: ~$0.62 per million characters.
    
    Kokoro voices (54 total):
      American English (af_*): af_heart, af_bella, af_sarah, af_sky, af_nicole
                    (am_*): am_adam, am_michael, am_echo, am_eric, am_liam
      British English  (bf_*): bf_emma, bf_isabella, bf_alice, bf_lily
                    (bm_*): bm_george, bm_lewis, bm_daniel, bm_fable
    """
    response = requests.post(
        "https://openrouter.ai/api/v1/audio/speech",
        headers={
            "Authorization": f"Bearer {openrouter_paid_api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": "hexgrad/kokoro-82m",
            "input": text,
            "voice": voice,
            "response_format": "mp3",
        },
    )

    response.raise_for_status()

    # Response is raw audio bytes, not JSON
    with open(output_file, "wb") as f:
        f.write(response.content)

    print(f"✅ Saved to {output_file} ({len(response.content) / 1024:.1f} KB)")
    return output_file


# Generate and play inline in Jupyter
audio_file = tts(
    text="Welcome to New York City! Home of the Statue of Liberty, Central Park, and the best pizza in the world.",
    voice="af_heart",
)

display(Audio(audio_file))

✅ Saved to speech.mp3 (47.8 KB)


In [56]:
def talker(message,voice="af_heart",output_file="speech.mp3"):
    response = requests.post(
        "https://openrouter.ai/api/v1/audio/speech",
        headers={
            "Authorization": f"Bearer {openrouter_paid_api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": "hexgrad/kokoro-82m",
            "input": message,
            "voice": voice,
            "response_format": "mp3",
        },
    )

    response.raise_for_status()

    # Response is raw audio bytes, not JSON
    with open(output_file, "wb") as f:
        f.write(response.content)

    print(f"✅ Saved to {output_file} ({len(response.content) / 1024:.1f} KB)")
    return output_file

In [ ]:
from openai import OpenAI
import requests
openrouter_key=os.getenv("OPENROUTER_API_KEY")
# Initialize client using OpenRouter configuration
client = OpenAI(
    base_url="https://openrouter.ai",
    
    api_key=openrouter_key,
)


print("Requesting image from OpenRouter...")

try:
    # Use OpenRouter's correct image generation endpoint
    response = client.images.generate(
        model="stabilityai/sd-xl", # Verified OpenRouter image model
        prompt="A beautiful cyberpunk garden with glowing floating plants, 8k resolution",
        n=1,
        size="1024x1024"
    )

    # 2. Extract the URL safely
    # OpenRouter returns a list-like data object containing the image URL
    image_url = response.data[0].url
    print(f"Image successfully generated on the cloud!")
    print(f"Direct URL link: {image_url}")

    # 3. Automatically download and save it to your machine
    print("Downloading image to your laptop...")
    image_data = requests.get(image_url).content
    with open("cyberpunk_garden.png", "wb") as f:
        f.write(image_data)
        
    print("Finished! Saved locally as 'cyberpunk_garden.png'")

except Exception as e:
    print(f"An error occurred: {e}")
    # If it failed, print the raw response to see what OpenRouter said
    print("Raw Response received:", response)

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [58]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [57]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [60]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Paris
DATABASE TOOL CALLED: Getting price for Tokyo
✅ Saved to speech.mp3 (76.7 KB)


# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>